# Stop reforking: where the ROGII error actually lives

*By [Georgy Mamarin](https://www.kaggle.com/georgymamarin)*

This notebook used to be called *Where the Wall Is*. There is no single wall. There is a part of the error you can recover from the data you are handed, and a tail you cannot model your way out of — and in the first version I had the irreducible tail mislabeled as the floor of the whole task.

Start with the numbers that anchor everything. The carry-last-TVT baseline scores **15.883** on the public board. The forks people keep reforking cluster around **7.2**. The board heads sit around **5.6**. The question worth asking before you fork anything: of that gap from 15.9 down to 5.6, how much is recoverable, and how much is geology?

The short version of the map:

- **Recoverable** — a per-well offset plus a piecewise dip, read by matching the horizontal well's gamma-ray to the typewell. The easier wells get to roughly 3–5&nbsp;ft per-well this way. Public forks (~7.2) have captured the offset and the dominant slope; the heads (~5.6) capture some of the wiggle on top.
- **Irreducible** — a bimodal datum on a minority of wells. The Eagle Ford is rhythmically bedded, so the GR can line up with the typewell at two stratigraphic positions about one bundle apart (roughly ±15&nbsp;ft). On those wells the truth is close to a coin-flip. In my train-side measurements, the worst ~10% of wells carry about 40% of the squared error.

Before the science, two corrections. I got two things wrong in the first version, and I will state them plainly.

**Correction 1 — there is no leak.** I claimed the public leaderboard was scored on three wells that are byte-for-byte copies of train wells, so the board read generously. It is not. That local `data/test/` folder is example data, replaced by the real hidden test (about 200 wells) at scoring — it says so on the Data page, and Ioannis M (rank 28) and the host posts confirm it. I conflated the example set with the scoring set. The public LB is about 26% of those ~200 hidden wells (on the order of 50), scored honestly; the private set decides medals; the host excluded one outlier private well. I retract the leak entirely. (For the curious: the example wells really are byte-identical to the first three train wells, and an ANCC-cheat scored about 0.007 against that visible example set, which is exactly what copies would do — that coincidence is what I misread. The detail lives in §10 now, not here.)

**Correction 2 — ~10&nbsp;ft is not a wall.** I framed ~10&nbsp;ft as the task's floor. It is not. The ~10 was *my selector's* ceiling — a per-well GR particle filter plus beam search — under a deliberately hard split that leaves whole fields out (block-CV ~10.9). Two things visible on the board already prove ~10 is not the floor, no inside information required: the heads score ~5.6, and my own smooth oracle (§2) sits at ~3.0. The leave-field-out negative result is real, but it answers a harder question (extrapolating to unseen *fields*) than the contest poses.

And the gap between my honest ~10 and the ~5.6 heads was never a leak. Tucker (rank 2) described reaching about 5 per-well, as I read his post, using per-well data only — so at least one head is simply matching GR better than my selector did. I do not know rank 1's method; the existence proof is enough. The gap is method quality.

This is a diagnostic, not a submission. It scores nothing, runs end-to-end on the public competition data alone, and is built to be forked — its honesty now rests on the real public-LB numbers (15.883 / ~7.2 / ~5.6), not on a leak.

One expectation, because it shaped how I wrote this: the most-upvoted notebook in the competition (a DWT one, 610 votes) scores about 9.25. On this task, votes follow clarity more than score. So treat this as a map of the problem, not a leaderboard lever.

## Contents
1. [The task in one paragraph](#1)
2. [The floor: a calibrated oracle ladder](#2)
3. [The error collapses to offset plus a piecewise dip](#3)
4. [Reading the dip: GR-to-typewell matching has a quality ceiling](#4)
5. [The irreducible tail: a bimodal datum](#5)
6. [The leave-field-out result, in proportion](#6)
7. [Three traps that look like progress](#7)
8. [What to do if you are stuck near the cluster](#8)
9. [Fork-and-go harness](#9)
10. [Design choices and limitations](#10)
11. [References and a question for you](#11)

<a id="setup"></a>
### Setup

Everything below runs on the **public competition data only** — no private artifacts — so a fork runs end-to-end. Seeded; a 250-well subsample keeps the run to a couple of minutes (the full 773 wells give the same picture).

In [1]:
import os, glob, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
warnings.filterwarnings("ignore")

SEED = 42; np.random.seed(SEED)
N_WELLS = 250                         # subsample for runtime; full 773 ~ same result
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True,
                     "grid.alpha": .25, "axes.axisbelow": True})
# colorblind-safe (Okabe–Ito)
C = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00",
         grey="#999999", purple="#CC79A7")

def find_train():
    for c in ["/kaggle/input/competitions/rogii-wellbore-geology-prediction/train",
              "/kaggle/input/rogii-wellbore-geology-prediction/train", "data/train"]:
        if os.path.isdir(c): return c
    g = glob.glob("/kaggle/input/**/train", recursive=True)   # robust mount resolve
    return g[0] if g else "data/train"

TRAIN = find_train()
files = sorted(glob.glob(f"{TRAIN}/*__horizontal_well.csv"))
rng = np.random.RandomState(SEED)
files = [files[i] for i in rng.permutation(len(files))[:N_WELLS]]
rmse = lambda a: float(np.sqrt(np.mean(np.asarray(a, float) ** 2)))
print(f"train dir: {TRAIN}")
print(f"wells in sample: {len(files)} / {len(glob.glob(f'{TRAIN}/*__horizontal_well.csv'))}")

<a id="1"></a>
## 1. The task in one paragraph

Each well is a horizontal trajectory with logs `MD, X, Y, Z, GR`, a known `TVT_input` up to a prediction-start point, and a vertical **typewell** (GR vs TVT). We predict `TVT` on the hidden eval tail; on the eval rows `X/Y/Z` are known and `TVT` is hidden. The metric is pooled row-wise RMSE over the eval rows.

A few community-established facts fix the problem; I take them as given.

The first is that the target reduces to a surface. `TVT = ANCC(surface) − Z + b_well`, with `b_well` constant per well (std 0.0066&nbsp;ft). `Z` is known on the tail and the offset is pinned by the heel, so predicting `TVT` is reading the formation surface along the lateral.

The second is that the six formation columns are not six surfaces. franticXu showed that `ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA` collapse to one base surface plus five constant offsets — one degree of freedom, with layer thickness std under 0.01&nbsp;ft along each well. They are derived from the typewell's TVT boundaries (in 97.5% of wells the typewell thickness equals the horizontal-well thickness), not measured 3D surfaces. One point that trips people up: `TVT` here is a *cumulative* vertical distance, like TVD, rather than the thickness of a single layer. The test wells ship without these formation columns, so they have to be reconstructed, and a few wells have a truncated or absent ANCC.

The third is what "matching" even means. Tabish's breakdown puts it cleanly: the typewell is a GR-vs-TVT lookup, and you read your stratigraphic position by matching the horizontal well's GR trace against it. The layers run roughly parallel, and many wells share subsequences of a master typewell sequence. So the target is not one number; it is a piecewise dip, what Tabish calls "parallel jagged lines."

The rest of the notebook maps where that surface is recoverable from the data and where it is not.

In [2]:
# load eval-row truth + geometry once; reused throughout
def load(f):
    hw = pd.read_csv(f)
    tvi, tv = hw["TVT_input"].to_numpy(float), hw["TVT"].to_numpy(float)
    Z, MD, GR = hw["Z"].to_numpy(float), hw["MD"].to_numpy(float), hw["GR"].to_numpy(float)
    X, Y = hw["X"].to_numpy(float), hw["Y"].to_numpy(float)
    ev = np.isnan(tvi) & np.isfinite(tv); ei = np.flatnonzero(ev)
    kn = np.flatnonzero(~np.isnan(tvi))
    if len(ei) < 50 or len(kn) < 50: return None
    li = kn[-1]
    s = (MD[ei] - MD[li]) / max(MD[-1] - MD[li], 1e-6)        # normalized tail position 0..1
    return dict(wid=os.path.basename(f)[:8], file=f, true=tv[ei], s=s, Z=Z[ei], MD=MD[ei],
                last=tvi[li], surf=tv[ei] + Z[ei], X=X, Y=Y, GR=GR, ei=ei, kn=kn, li=li,
                tvi=tvi, hwZ=Z, hwMD=MD)
W = [w for w in (load(f) for f in files) if w is not None]
print(f"usable wells: {len(W)}")

<a id="2"></a>
## 2. The floor: a calibrated oracle ladder

Before chasing a better model, ask the cheaper question: if you already knew the answer, what is the best each modeling assumption could do? Each rung below uses the *true* tail to fit the simplest possible shape, then reports its pooled RMSE. That is a ceiling no model *restricted to that shape* can beat — and the ladder turns out to be anchored to the real board, which is the point.

- **flat** — carry the last known TVT (the only fully-legal rung here; a sanity floor).
- **constant** — the best single per-well offset.
- **line** — the best per-well straight line vs measured depth.
- **smooth** — a robust degree-6 fit (captures the wiggle too).

In [3]:
def robfit(s, y, deg):
    if len(s) < deg + 2: return y.copy()
    c = np.polyfit(s, y, deg)
    for _ in range(3):
        r = y - np.polyval(c, s); sc = np.median(np.abs(r)) * 1.4826 + 1e-6
        c = np.polyfit(s, y, deg, w=1.0 / (1.0 + (r / (2 * sc)) ** 2))
    return np.polyval(c, s)

E = {k: [] for k in ["flat", "constant", "line", "smooth"]}
for w in W:
    t, s = w["true"], w["s"]
    E["flat"].append(t - w["last"])
    E["constant"].append(t - t.mean())
    E["line"].append(t - np.polyval(np.polyfit(s, t, 1), s))
    E["smooth"].append(t - robfit(s, t, 6))
ladder = {k: rmse(np.concatenate(v)) for k, v in E.items()}
_tbl = pd.DataFrame({"Oracle RMSE (ft)": [ladder[k] for k in ["flat","constant","line","smooth"]],
                     "What it captures": ["carry the last known value", "best single offset", "best straight line", "adds the wiggle"]},
                    index=["flat (legal)", "constant offset", "per-well line", "smooth (deg-6)"])
_tbl.index.name = "Assumption (oracle)"

# per-well figures used later (measured here, not borrowed):
line_w   = np.array([rmse(e) for e in E["line"]])      # per-well RMSE under the line oracle
smooth_w = np.array([rmse(e) for e in E["smooth"]])    # ...and under the smooth oracle
sse_w    = np.array([float(np.sum(e**2)) for e in E["smooth"]])
_ord = np.argsort(sse_w)[::-1]; _k10 = max(1, len(sse_w)//10)
worst10_share = sse_w[_ord[:_k10]].sum() / sse_w.sum()
print(f"wells already under 5 ft per-well:  line oracle {np.mean(line_w<5):.0%}   smooth oracle {np.mean(smooth_w<5):.0%}")
print(f"worst 10% of wells carry {worst10_share:.0%} of the smooth-oracle squared error  (the tail; see §5)")
_tbl.style.format({"Oracle RMSE (ft)": "{:.2f}"}).background_gradient(subset=["Oracle RMSE (ft)"], cmap="Greens_r")

**Table 1 — the oracle ladder.** Pooled RMSE over 250 train wells; each rung fits the simplest shape to the *true* tail, so it is an unreachable ceiling *for that assumption*, not a model score.

In [4]:
fig, ax = plt.subplots(figsize=(7.8, 3.6))
names = ["flat\n(legal)", "constant\noffset", "per-well\nline", "smooth\n(deg-6)"]
vals = [ladder[k] for k in ["flat", "constant", "line", "smooth"]]
bars = ax.bar(names, vals, color=[C["grey"], C["orange"], C["blue"], C["green"]], width=.62)
for b, v in zip(bars, vals): ax.text(b.get_x()+b.get_width()/2, v+.15, f"{v:.1f}", ha="center", fontsize=10)
# reference lines — CITED from the REAL public LB (honestly scored on the hidden wells), NOT recomputed here.
ax.axhspan(5.4, 7.4, color="#cccccc", alpha=.18, zorder=0)
ax.axhline(15.883, ls="--", lw=1.3, color=C["grey"],   label="carry-last baseline 15.9  (public LB)")
ax.axhline(7.2,    ls="--", lw=1.3, color=C["red"],    label="public best fork ~7.2  (public LB)")
ax.axhline(5.6,    ls="--", lw=1.3, color=C["purple"], label="LB heads ~5.6  (public LB)")
ax.legend(loc="upper right", fontsize=8.5, framealpha=.95)
ax.text(0.02, 0.55, "bars: train wells, pooled RMSE\ndashed: real public LB, cited",
        transform=ax.transAxes, va="top", ha="left", fontsize=8, color="#333",
        bbox=dict(boxstyle="round,pad=0.3", fc="#f7f7f7", ec="#bbbbbb", alpha=.95))
ax.set_ylabel("RMSE (ft, lower = better)"); ax.set_ylim(0, 17.5)
ax.set_title("Oracle ladder (train, pooled RMSE)  vs  the real public LB")
plt.tight_layout(); plt.show()

**Read it like this.** Carrying the last value is hopeless (~16). The live carry-last baseline scores **15.883**. Same procedure, different well sets — my flat oracle on a 250-well train sample, the baseline on the hidden public wells — so I read the agreement as a sanity check that my ladder is on the board's scale, not as proof of anything finer.

From there it descends. One offset per well already cuts it to ~9. The per-well **line** oracle (~6.6) sits just below the fork cluster (~7.2). I read that proximity as the forks capturing close to the offset-plus-line content and not much past it, but that is an inference from where the scores land, not a decomposition of what the forks actually do. The board **heads** at ~5.6 reach *below* the line oracle, which is the cleanest evidence in the chart that they are modeling more than a line — some of the wiggle and the dip changes. And the smooth oracle at ~3.0 says there is still recoverable structure sitting under even the heads.

One fence, so the two kinds of marker are not confused. The **bars** are train-fit ceilings — pooled RMSE on 250 train wells, each unreachable for its assumption because each needed the truth to fit. The **dashed lines** are live, honestly-scored public-LB markers. Different categories, same RMSE scale. The ordering is what carries the argument: forks near the line oracle, heads reaching into the wiggle, recoverable structure still below. The next sections are about what "reaching into the wiggle" requires.

<a id="3"></a>
## 3. The error collapses to offset plus a piecewise dip

Take the surface along each lateral, `ANCC = TVT + Z`, and fit a straight line in measured depth. How much is left?

In [5]:
lin_r2, wiggle, slopes = [], [], []
for w in W:
    surf, s = w["surf"], w["s"]
    lin = np.polyval(np.polyfit(s, surf, 1), s)
    ss_res = np.sum((surf - lin) ** 2); ss_tot = np.sum((surf - surf.mean()) ** 2)
    lin_r2.append(1 - ss_res / max(ss_tot, 1e-9))
    wiggle.append(np.std(surf - lin))
    slopes.append(np.polyfit(w["s"], w["true"] - w["last"], 1)[0])   # drift slope vs flat baseline
print(f"surface linear-fit R^2 : median {np.median(lin_r2):.3f}   ({np.mean(np.array(lin_r2)>0.9):.0%} of wells > 0.90)")
print(f"residual wiggle (std)  : median {np.median(wiggle):.2f} ft")

In [6]:
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.3))
ax[0].hist(lin_r2, bins=30, color=C["blue"], alpha=.85)
ax[0].axvline(np.median(lin_r2), color=C["red"], lw=1.5)
ax[0].set_title("Surface is a line"); ax[0].set_xlabel("per-well linear-fit R²  (ANCC vs MD)")
ax[0].set_ylabel("wells"); ax[0].text(.05,.9,f"median {np.median(lin_r2):.3f}",transform=ax[0].transAxes,color=C["red"])
ax[1].hist(np.clip(wiggle,0,15), bins=30, color=C["green"], alpha=.85)
ax[1].axvline(np.median(wiggle), color=C["red"], lw=1.5)
ax[1].set_title("…and the wiggle around it is small"); ax[1].set_xlabel("residual after the line (ft, std)")
ax[1].set_ylabel("wells"); ax[1].text(.4,.9,f"median {np.median(wiggle):.1f} ft",transform=ax[1].transAxes,color=C["red"])
plt.tight_layout(); plt.show()

For the typical well the surface is a straight line to within a few feet over the whole lateral: median R² ≈ 0.99. The exceptions are the faulted minority, with steps of 15–30&nbsp;ft, and they are exactly the piecewise tail §5 is about, so don't let the median hide them. The line claim is about the middle of the distribution, not its ends.

That collapses the problem, but not down to a single number. Once the per-well offset is set by the heel, what is left is the dip, and the dip varies along the well. Following Tabish, `TVT + Z` and the layers are piecewise continuous linear functions, the "parallel jagged lines": a dominant slope plus the break points where it changes. The offset is pinned by the heel; the dip and its breaks are what you have to read. The breaks come from the same sub-seismic faults — they belong to the hard tail (§5), not to noise you should smooth away.

This is the recoverable part of the map. The offset plus the dominant slope is what the public forks (~7.2) have captured; the heads (~5.6) pull in some of the dip changes and the wiggle on top. The cell in §2 prints how many wells already sit under 5&nbsp;ft per-well from a clean smooth fit — a sizable chunk, which is why the room to improve is on the median wells. So the gap below the cluster is a matching problem: read the dip off the typewell well enough, on more wells. The next section is about that match.

<a id="4"></a>
## 4. Reading the dip: GR-to-typewell matching has a quality ceiling

Geosteering works because the gamma-ray log fingerprints the rock: match the well's `GR` to the typewell's `GR`-vs-`TVT` profile and you know your vertical position. The question is how far that match gets you, and what the bottleneck is.

First, the legality, because it decides what "matching" is even allowed to use. Settled in the Tiago thread: using the full per-well GR trace, including rows ahead of the eval row (the within-well "look-ahead"), is legal. This is post-drilling analysis, and the rule there was that everything in the data we are given is fair game unless explicitly forbidden. So matching your whole GR trace against the typewell, which is what the top solvers do, is fair.

And it reaches further than I gave it credit for. Tucker (rank 2) described reaching about 5 per-well using per-well data only — no cross-well features, no spatial structure, no external tops. I am reading that off his post rather than a shared notebook, so I will not lean on it harder than that. But even as one data point it scopes the task: for at least the front of the field, the live lever is per-well GR-matching quality, and the cross-field slope transfer I spent a section proving hard (§6) is not required to get there.

So why doesn't the match trivially nail it? There is a real degeneracy, and I measure it with a **shift scan**. For each well, slide a candidate vertical offset `dz` and ask where the GR-misfit against the typewell is smallest, under three calibrations of the GR gain and offset:

- **oracle** — calibrate at the *true* TVT (illegal; an upper bound on what GR knows).
- **legal** — calibrate at a legitimate flat-surface guess (`last_known + (Z_heel − Z)`, known quantities only).
- **shuffle** — calibrate at a shuffled TVT (a random control).

In [7]:
GRID = np.arange(-20.0, 20.01, 0.5)            # candidate vertical shifts dz; 0 = the truth
def twgr(tvt, tv, tg): return np.interp(tvt, tv, tg, left=tg[0], right=tg[-1])
def load_tw(w):
    twf = w["file"].replace("__horizontal_well.csv", "__typewell.csv")
    if not os.path.exists(twf): return None, None
    tw = pd.read_csv(twf); tv = tw["TVT"].to_numpy(float); tg = tw["GR"].to_numpy(float)
    o = np.argsort(tv); tv, tg = tv[o], tg[o]; _, ix = np.unique(tv, return_index=True)
    return tv[ix], tg[ix]
def misfit_curve(gr, true, calib_tvt, tv, tg):
    v = np.isfinite(gr)
    tgt = twgr(calib_tvt, tv, tg)
    coef, *_ = np.linalg.lstsq(np.vstack([gr[v], np.ones(v.sum())]).T, tgt[v], rcond=None)  # fit gain/offset
    cal = gr * coef[0] + coef[1]
    return np.array([np.mean((cal[v] - twgr(true + dz, tv, tg)[v]) ** 2) for dz in GRID])
def argmin_dz(gr, true, calib_tvt, tv, tg):
    return abs(GRID[int(np.argmin(misfit_curve(gr, true, calib_tvt, tv, tg)))])

orc, leg, shf, lat_span, tw_span, worst_idx = [], [], [], [], [], []
rs = np.random.RandomState(SEED)
for wi, w in enumerate(W):
    tv, tg = load_tw(w)
    if tv is None: continue
    gr = pd.Series(w["GR"][w["ei"]]).interpolate(limit_direction="both").to_numpy()
    true = w["true"]
    if np.sum(np.isfinite(gr)) < 20 or len(tv) < 10 or np.std(gr) < 1e-6: continue
    legal = w["last"] + (w["hwZ"][w["li"]] - w["Z"])          # legal flat-surface guess; Z is known on the tail (§1)
    orc.append(argmin_dz(gr, true, true, tv, tg))
    leg.append(argmin_dz(gr, true, legal, tv, tg))
    shf.append(argmin_dz(gr, true, true[rs.permutation(len(true))], tv, tg))
    lat_span.append(np.percentile(true,97)-np.percentile(true,3)); tw_span.append(np.percentile(tv,97)-np.percentile(tv,3))
    worst_idx.append(wi)
orc, leg, shf = map(np.array, (orc, leg, shf))
from IPython.display import display
_deg = pd.DataFrame({"median |dz| from truth (ft)": [np.median(orc), np.median(leg), np.median(shf)],
                     "localized within 2 ft": [np.mean(orc<=2), np.mean(leg<=2), np.mean(shf<=2)]},
                    index=["oracle (illegal)", "legal (public)", "shuffle (control)"])
_deg.index.name = "GR calibration"
display(_deg.style.format({"median |dz| from truth (ft)": "{:.2f}", "localized within 2 ft": "{:.0%}"})
        .apply(lambda r: ["background-color:#fde7e0"]*len(r) if r.name == "legal (public)" else [""]*len(r), axis=1))
print(f"context: the answer moves ~{np.median(lat_span):.0f} ft through a ~{np.median(tw_span):.0f} ft typewell column")

**Table 2 — the calibration degeneracy.** Median |dz| from the true surface and the fraction of wells localized within 2 ft, under three GR calibrations; the legal row (highlighted) sits at the random-shuffle level — the explicit gain/offset fit localizes only when calibrated at the truth.

In [8]:
fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.4))
labs = ["oracle\n(illegal)", "legal\n(public)", "shuffle\n(control)"]
frac = [np.mean(orc<=2), np.mean(leg<=2), np.mean(shf<=2)]
bars = ax[0].bar(labs, [f*100 for f in frac], color=[C["green"], C["red"], C["grey"]], width=.6)
for b, f in zip(bars, frac): ax[0].text(b.get_x()+b.get_width()/2, f*100+2, f"{f:.0%}", ha="center")
ax[0].set_ylabel("wells localized within 2 ft (%)"); ax[0].set_ylim(0, 100)
ax[0].set_title("The explicit fit pins the surface — only at the true calibration")
ax[1].barh(["typewell\ncolumn", "lateral\nanswer band"], [np.median(tw_span), np.median(lat_span)],
           color=[C["grey"], C["blue"]])
for i, v in enumerate([np.median(tw_span), np.median(lat_span)]): ax[1].text(v+10, i, f"{v:.0f} ft", va="center")
ax[1].set_xlabel("TVT span (ft)"); ax[1].set_title("…and the lateral barely moves through it")
plt.tight_layout(); plt.show()

Read the left panel. With the **oracle** calibration the GR-misfit minimum lands on the true surface in about 82% of wells (median offset 0 ft). The gamma-ray tracks lithology, as it should. But fit the same explicit gain/offset at any **legal** position and that skill collapses: only ~8% of wells localize, at the random-shuffle level, with the minimum wandering a dozen feet. The right panel says why this particular fit struggles. The answer moves through about 25&nbsp;ft while the typewell spans hundreds, so the lateral never travels far enough vertically to pin the gain and offset from the data alone.

So the correction to my earlier claim. That confound is a property of *this explicit-calibration method*, not of the data. GR carries real coarse signal — a particle filter takes my selector from the ~16 floor down to ~10 with it (~10.9 under leave-whole-field-out), which is that tracker's level under a hard split, not a GR ceiling. And Tucker's ~5 per-well, as I read it, says a better matcher pulls more of the dip and wiggle out of the same data. The gap from my ~10 to ~5 is alignment headroom, not an information wall.

So the earlier framing — "GR cannot pin the slope, and that is a property of the data" — was too strong. Read this section as: the naive explicit gain/offset fit is confounded with position, and my tracker's ~10&nbsp;ft is *that tracker's* limit under one hard split. A sequential matcher that uses the typewell directly, the way the ~5&nbsp;ft solvers do, sidesteps the explicit fit and gets further.

One practical lever before the next section, from Tabish: the GR sensor rotates as the tool turns in the hole, and an FFT of the GR shows the rotation frequency. You can use that to denoise the trace before matching — cleaner input, better alignment.

<a id="5"></a>
## 5. The irreducible tail: a bimodal datum

If the recoverable part is offset plus dip read off the typewell, the irreducible part is where the typewell stops telling you the truth. This is the piece I think matters most.

The geology comes from souldrive, whose post laid it out (and who, I'll note, publicly endorsed the first version of this notebook — credit due both ways). The Eagle Ford is rhythmically bedded: Milankovitch orbital cyclicity drives limestone–marl couplets that repeat on a ~15–25&nbsp;ft scale. Because the bedding repeats, the horizontal GR can line up with the typewell at two stratigraphic positions about one bundle apart. The datum becomes bimodal, roughly ±15&nbsp;ft, and on a truly ambiguous well it is close to a coin-flip which mode is real. A minority of wells add their own version of this through sub-seismic faults, 15–30&nbsp;ft steps.

I came at the same object from the other side, and the measurement is in the cell below, computed on public train so a forker can reproduce it. Fit the smooth oracle per well, take each well's squared error, and sort: the worst decile of wells carries close to 40% of the total (the cell prints the exact figure for the sample). That is the concentration souldrive's geology predicts. Reusing the §4 shift scan on those worst-decile wells, under the *oracle* calibration — the one that did localize 82% of normal wells — the GR misfit near the true TVT is flat, or slightly worse than a match one bundle away. Because this is the calibration that works on normal wells, a flat misfit here is the data being ambiguous, not the calibration failing. The typewell cannot distinguish the two positions on those wells.

The decision theory is what makes this load-bearing, and pilkwang sharpened it in the comments. On a well with mass `p` at one position and `1−p` at the other a distance `d` apart, the squared-loss-optimal estimate is the posterior mean — `p·a + (1−p)·b`, with irreducible risk `p(1−p)d²`. The midpoint I first reached for is just the `p = ½` case. Committing to a single mode beats it only outside `¼ < p < ¾`, while the posterior mean does at least as well as either for every `p` — strictly better than the midpoint whenever `p ≠ ½`, and strictly better than a committed mode for every `p` interior to (0,1). So the target isn't a binary "is this well bimodal" flag feeding a fixed midpoint — it's a calibrated `p`, and the estimate then leans toward whichever mode `p` favors.

And `p` falls out of machinery already on the page: the *relative depth* of the two minima in the §4 shift scan is a log-likelihood ratio between the modes, so a two-position softmax over those depths gives `p` directly. Two things I'd add to his recipe. First, the softmax needs a calibrated temperature or `p` comes out overconfident. Second — the wrinkle I keep hitting — on the worst-decile wells the GR likelihood is flat or slightly inverted at the truth: the decoy a bundle away fits the GR a touch better, so a `p` read straight off the misfit is biased toward the wrong mode. That bias is probably why the tail is sticky rather than merely uncertain. The irreducible loss is the variance of the datum; the practical question is how well you can estimate `p` before it bites.

Stack §3 through §5 together and the single "wall" dissolves into a distribution. A good chunk of wells are near-solved from clean offset plus dip (the cell in §2 prints how many sit under 5&nbsp;ft per-well). The heavy tail is geology — the bimodal datum and the faults — not a modeling failure. That recoverable-versus-irreducible split is the honest replacement for the ~10&nbsp;ft wall I claimed before.

A note on what is measured versus borrowed, so I don't over-correct one mistake into a new overclaim. The Milankovitch mechanism is souldrive's geology. The SSE concentration and the likelihood degeneracy are my train-side measurements, computed on the smooth-oracle residuals in the cell below; they need the truth, so they live on train, not on the hidden test. The midpoint-versus-mode arithmetic is decision theory on an idealized well. I am not claiming to have measured the bimodal *cause* on every tail well. The ±15&nbsp;ft picture beside the figure is a labeled schematic of the mechanism, not a measurement.

In [9]:
# ---- §5, train-confirmed: where the error concentrates, and why those wells are ambiguous ----
order = np.argsort(sse_w)[::-1]                 # wells worst-first, by smooth-oracle SSE (from §2)
k10 = max(1, len(sse_w)//10)
share = sse_w[order[:k10]].sum() / sse_w.sum()

fig, ax = plt.subplots(1, 3, figsize=(13.6, 3.7))

# (a) cumulative-SSE (Lorenz) curve — the concentration, measured on train
cum  = np.cumsum(sse_w[order]) / sse_w.sum()
xf   = np.arange(1, len(cum)+1) / len(cum)
ax[0].plot(xf, cum, color=C["red"], lw=2)
ax[0].plot([0,1],[0,1], ls="--", lw=1, color=C["grey"], label="if error were uniform")
ax[0].scatter([0.10],[share], color=C["blue"], zorder=5)
ax[0].annotate(f"worst 10% of wells\ncarry {share:.0%} of SSE", (0.10, share),
               xytext=(0.34, max(share-0.22, 0.12)), fontsize=9,
               arrowprops=dict(arrowstyle="->", color=C["blue"]))
ax[0].set_xlabel("fraction of wells (worst first)"); ax[0].set_ylabel("cumulative share of SSE")
ax[0].set_title("Error is concentrated (train-confirmed)"); ax[0].legend(loc="lower right", fontsize=8)

# (b) GR misfit vs dz under the ORACLE calibration, for the worst few wells — truth not preferred
shown = 0
for wi in order:
    if shown >= 3: break
    w = W[wi]; tv, tg = load_tw(w)
    if tv is None: continue
    gr = pd.Series(w["GR"][w["ei"]]).interpolate(limit_direction="both").to_numpy()
    if np.sum(np.isfinite(gr)) < 20 or len(tv) < 10 or np.std(gr) < 1e-6: continue
    c = misfit_curve(gr, w["true"], w["true"], tv, tg)      # oracle calibration (at the truth)
    ax[1].plot(GRID, c / c.min(), lw=1.5, alpha=.85, label=w["wid"])
    shown += 1
ax[1].axvline(0, color="k", lw=.9, ls="--")
ax[1].set_xlabel("vertical shift dz from truth (ft)"); ax[1].set_ylabel("GR misfit (÷ its own min)")
ax[1].set_title("Worst wells: the truth isn't preferred"); ax[1].legend(fontsize=8, title="well")

# (c) schematic of the bimodal datum — an illustration of souldrive's mechanism, NOT a measurement
xx = np.linspace(-35, 35, 400); g = lambda m: np.exp(-0.5*((xx-m)/4.0)**2)
ax[2].fill_between(xx, g(-15), color=C["orange"], alpha=.55, label="mode A")
ax[2].fill_between(xx, g(15),  color=C["blue"],   alpha=.40, label="mode B (~1 bundle away)")
ax[2].axvline(0, color=C["green"], lw=2.2, label="posterior mean (midpoint at p=½)")
ax[2].set_yticks([]); ax[2].set_xlabel("TVT error (ft)")
ax[2].set_title("Schematic: bimodal datum (±15 ft)"); ax[2].legend(fontsize=7.5, loc="upper right")
plt.tight_layout(); plt.show()
print(f"worst 10% of wells carry {share:.0%} of the smooth-oracle squared error (train sample of {len(W)} wells)")

**Does the hedge actually help? I tested pilkwang's recipe.** Centre the §4 shift scan on the truth and slide a constant datum offset — this centres on the truth, so it is a train-only diagnostic that measures whether the hedge *could* help, not a procedure you can run on the test set. The two deepest minima are the two candidate datums, and a two-position softmax over their depths gives `p`. The cell scores three datums on every well — committing to the global minimum, the posterior mean, and the fixed midpoint — under the *oracle* calibration, since §4b already showed the legal explicit fit barely localizes. The number is the constant datum error each leaves (0 = perfect; the bimodal gap is ~±15&nbsp;ft).

In [10]:
def two_minima(cost, sep=6.0):
    i1 = int(np.argmin(cost)); dz1, m1 = GRID[i1], cost[i1]
    mask = np.abs(GRID - dz1) >= sep
    if not mask.any(): return dz1, m1, dz1, m1
    j = np.flatnonzero(mask)[int(np.argmin(cost[mask]))]
    return dz1, m1, GRID[j], cost[j]
def softmax_p(m1, m2, T):
    a = np.array([-m1 / T, -m2 / T]); a -= a.max(); e = np.exp(a); return e[0] / e.sum()

tail_set = set(order[:k10].tolist())                       # worst-decile wells, from the SSE sort above
err = {m: ([], []) for m in ["commit", "posterior", "midpoint"]}
loc_oracle = loc_legal = wrong = n = 0
for wi, w in enumerate(W):
    tv, tg = load_tw(w)
    if tv is None: continue
    gr = pd.Series(w["GR"][w["ei"]]).interpolate(limit_direction="both").to_numpy()
    if np.sum(np.isfinite(gr)) < 20 or len(tv) < 10 or np.std(gr) < 1e-6: continue
    legal = w["last"] + (w["hwZ"][w["li"]] - w["Z"])       # legal flat-surface guess (as in §4b)
    c_or = misfit_curve(gr, w["true"], w["true"], tv, tg)  # oracle calibration; dz=0 is the truth
    c_lg = misfit_curve(gr, w["true"], legal, tv, tg)      # legal calibration
    n += 1
    dz1, m1, dz2, m2 = two_minima(c_or)
    if abs(dz1) <= 2: loc_oracle += 1
    if abs(two_minima(c_lg)[0]) <= 2: loc_legal += 1
    if abs(dz1) > abs(dz2): wrong += 1
    p1 = softmax_p(m1, m2, 2.0 * m1 + 1e-9)            # T = 2*min_misfit ≈ 2σ², the Bayesian temperature
    pred = dict(commit=dz1, posterior=p1 * dz1 + (1 - p1) * dz2, midpoint=0.5 * (dz1 + dz2))
    nev = len(w["true"])
    for m in err:
        err[m][0].extend([pred[m]] * nev)
        if wi in tail_set: err[m][1].extend([pred[m]] * nev)
datum = {m: (rmse(err[m][0]), rmse(err[m][1])) for m in err}
print("datum error left (ft, oracle calibration):     all wells | worst-decile")
for m in ["commit", "posterior", "midpoint"]:
    print(f"  {m:<11} {datum[m][0]:>6.2f}     {datum[m][1]:>6.2f}")
print(f"\nglobal minimum localizes <=2 ft:  oracle {loc_oracle/n:.0%}   legal {loc_legal/n:.0%}"
      f"   (under oracle it still picks the wrong mode {wrong/n:.0%})")

In [11]:
fig, ax = plt.subplots(figsize=(6.6, 3.3))
labs = ["commit", "posterior\n(pilkwang)", "midpoint"]
allv = [datum[m][0] for m in ["commit", "posterior", "midpoint"]]
tlv  = [datum[m][1] for m in ["commit", "posterior", "midpoint"]]
x = np.arange(3)
ax.bar(x-0.19, allv, 0.36, color=C["grey"], label="all wells")
ax.bar(x+0.19, tlv,  0.36, color=C["red"],  label="worst-decile tail")
for i, (a, t) in enumerate(zip(allv, tlv)):
    ax.text(i-0.19, a+0.08, f"{a:.1f}", ha="center", fontsize=8.5)
    ax.text(i+0.19, t+0.08, f"{t:.1f}", ha="center", fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels(labs); ax.set_ylabel("datum error left (ft)")
ax.set_title("The hedge vs committing (oracle calibration)"); ax.legend(fontsize=8.5)
plt.tight_layout(); plt.show()

On the worst-decile wells (25 of them) the posterior mean leaves the least datum error of the three — it beats both committing and the fixed midpoint, so pilkwang's calibrated hedge helps exactly where it should, on the ambiguous tail. On the easy wells the global minimum is already right, so committing there is fine; averaged over all wells committing actually wins (5.2&nbsp;ft vs 5.7&nbsp;ft for the posterior mean), because the Bayesian temperature over-hedges the many easy wells where the global min is already correct. The hedge's value is concentrated on the tail, which carries most of the error anyway. The tail margin is only ~0.7&nbsp;ft, though, so call it directional. Two caveats. The softmax `p` is a confidence weight, not a mode-fixer: it always backs the lower-misfit minimum. When that minimum is the decoy (12% of wells even under the oracle calibration), `p` backs the decoy too — the gain comes from not fully committing, not from `p` choosing the right mode. And this is the *oracle*-calibration ceiling: the global min localizes within 2&nbsp;ft on 82% of wells under the oracle calibration but only 8% under the legal flat fit, so the legal route is a sequential matcher carrying the multimodal posterior, not this explicit fit. The hedge is real; the open problem is the calibration.

<a id="6"></a>
## 6. The leave-field-out result, in proportion

This is the experiment that led me to the wrong "wall" claim, so let me run it honestly and then scope it correctly.

The setup is conservative by design. Cluster the wells by location (KMeans on heel X/Y), then GroupKFold so each fold leaves whole fields out, and try to predict the per-well drift slope from cheap legal features. The result: field-grouped OOF R² below zero — worse than guessing the global mean — and the shuffled-label control sits in the same place, so the harness itself is not leaking. CNN, GRU and GBM all failed to beat the selector. Cross-well surface reconstruction (IDW / plane fit, leave-field-out) failed badly: 13–129&nbsp;ft.

The finding is real: the per-well drift slope is not learnable from the cheap legal features I tried when you generalize to unseen fields. I stand by that.

What I got wrong was the scope. This is leave-*whole-field*-out, deliberately the hardest split I could pose. The competition does not ask you to extrapolate to brand-new fields, and I should not assume more about the test geometry than the host has confirmed. My selector's honest level was ~10 pooled (~10.9 under this block-CV) — its ceiling under one hard split, not the task's floor. Tucker's ~5 per-well, as I read it, goes well below that without ever touching the cross-field slope transfer I proved hard.

So state the boundary cleanly, because both facts are true and answer different questions. The negative result says cross-field slope transfer is hard from cheap features; that is real. It does not say the per-well dip is unreadable — the board heads at ~5.6 and my own smooth oracle at ~3.0 say the opposite — and it is not a within-field learnability claim either way.

The useful caution survives intact: a within-sample slope fit looks predictive and evaporates on hold-out, so "I found a feature that predicts the drift" usually means "I leaked." Still true. It just was not the floor of the task.

In [12]:
# per-well legal features (known at prediction time) + the target slope
rows = []
for w in W:
    li, kn = w["li"], w["kn"]; Zf, MDf, X, Y = w["hwZ"], w["hwMD"], w["X"], w["Y"]
    ancc_kn = w["tvi"][kn] + Zf[kn]; latd_kn = np.hypot(X[kn]-X[li], Y[kn]-Y[li])
    pre_dip = np.polyfit(latd_kn, ancc_kn, 1)[0] if latd_kn.max() > 50 else 0.0
    g = lambda a: np.gradient(a, MDf)[li]
    rows.append(dict(wid=w["wid"], x=X[li], y=Y[li],
                     slope=np.polyfit(w["s"], w["true"]-w["last"], 1)[0],
                     pre_dip=pre_dip, dz_heel=g(Zf),
                     incl=np.degrees(np.arctan2(np.hypot(g(X), g(Y)), abs(g(Zf))+1e-9)),
                     gr_pre=np.nanmean(w["GR"][kn][-50:]),
                     latspan=np.hypot(X[w["ei"][-1]]-X[li], Y[w["ei"][-1]]-Y[li])))
D = pd.DataFrame(rows)
N_FIELDS = 17                                          # match the cited block-CV (KMeans-17 on heel X/Y)
XY = D[["x","y"]].values; XYs = (XY-XY.mean(0))/(XY.std(0)+1e-9)
field = KMeans(N_FIELDS, n_init=10, random_state=SEED).fit(XYs).labels_   # spatial field clusters
feats = ["pre_dip","dz_heel","incl","gr_pre","latspan","x","y"]
y = D["slope"].values
r2 = lambda yt,yp: 1 - np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2)
oof, oof_shuf = np.zeros(len(D)), np.zeros(len(D)); yshuf = rng.permutation(y)
for k in range(N_FIELDS):
    tr, te = field!=k, field==k
    if te.sum()==0 or tr.sum()<15: oof[te]=oof_shuf[te]=y[tr].mean() if tr.sum() else y.mean(); continue
    mk = lambda yy: HGB(max_iter=250,max_depth=3,learning_rate=.05,l2_regularization=2.,random_state=SEED).fit(D.loc[tr,feats],yy[tr]).predict(D.loc[te,feats])
    oof[te], oof_shuf[te] = mk(y), mk(yshuf)
print(f"drift-slope field-OOF R² = {r2(y,oof):+.3f}")
print(f"shuffle-control     R² = {r2(y,oof_shuf):+.3f}   (≈0 ⇒ the harness itself does not leak)")

In [13]:
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.5))
ax[0].scatter(y, oof, s=14, alpha=.6, color=C["blue"])
lim = np.percentile(np.abs(y), 98)
ax[0].plot([-lim,lim],[-lim,lim], color=C["grey"], ls="--", lw=1, label="ideal (y = x)")
ax[0].set_xlim(-lim,lim); ax[0].set_ylim(-lim,lim)
ax[0].set_xlabel("true drift slope"); ax[0].set_ylabel("predicted (field hold-out)")
ax[0].legend(loc="upper left", fontsize=8, framealpha=.9)
ax[0].set_title(f"No skill on unseen fields  (R²={r2(y,oof):+.2f})")
labels = ["drift slope\n(field-OOF)", "shuffle\ncontrol"]
ax[1].bar(labels, [r2(y,oof), r2(y,oof_shuf)], color=[C["red"], C["grey"]], width=.5)
ax[1].axhline(0, color="k", lw=.8); ax[1].set_ylabel("R²  (>0 ⇒ learnable)")
ax[1].set_title("Both sit at/below zero")
plt.tight_layout(); plt.show()

<a id="7"></a>
## 7. Three traps that look like progress

Three things on this task feel like progress without being it. The first two are about how results are read; the third I walked into myself.

**The CV→LB mirage.** A deep model can post a low validation RMSE on train folds and land far worse when it runs for real. At least one public notebook here advertises a single-digit fold RMSE yet scores around 14 when actually run. A low cross-validation number is a claim about *seen* wells; the leaderboard is a claim about *unseen* ones, and on this task they come apart hard. Validate on a hold-out shaped like the real test, not the friendliest fold.

**Seed and refork variance.** The strong public pipeline contains a stochastic particle filter. Re-running the same code, or forking a notebook whose public score is a few thousandths better, draws a fresh sample, and the run-to-run spread is comparable to the fork-to-fork gaps people chase (the real cluster lives around 7.15–7.5). pilkwang put a clean number on this in the comments: his byte-identical notebooks scored 7.201–7.286 on the public board, and a wider set identical up to comment encoding reaches down to 7.168 — the whole spread is the particle filter reseeding, with nothing changed in the code. So "a fork that reads 0.03 better than its neighbor" is just the lucky tail of that spread, and picking your final submission by best public score is selecting on it. Before celebrating a small delta, check whether it clears the seed band.

**Chasing the ambiguous tail.** This is the counterintuitive one. On a bimodal well (§5), committing to a mode — the move that feels like finally *solving* that well — raises your expected error versus the midpoint hedge (about 21 vs 15 on the idealized 50/50 well). So apparent progress on the hardest wells can run in the wrong direction. Spend the effort where the error is recoverable, and hedge the tail.

And a meta-trap, since I fell into it: a deliberately conservative validation split can over-reject. My leave-field-out told me "unlearnable," and I generalized that to "the task's floor." Match the split to the test the competition actually scores, or a pessimistic guardrail turns into a false ceiling.

<a id="8"></a>
## 8. What to do if you are stuck near the cluster

- **The leverage is the median wells, not the tail.** Clean structural smoothing of the offset plus piecewise dip, and better per-well GR alignment to the typewell — that is where ~7.2&nbsp;→&nbsp;~5.6 lives, and it is per-well, not cross-field. A sizable share of wells can reach 3–5&nbsp;ft (the §2 cell prints how many).
- **Estimate the mode weight, predict the posterior mean.** On a bimodal well the variance is irreducible, so the right estimate is `p·a + (1−p)·b`, not a commitment to one mode (§5). The detector and the hedge collapse into one object: read `p` via the two-position softmax over the relative depths of the two minima in the §4 shift scan, as in §5 — but note that under the legal calibration that scan localizes only weakly (§4), so the depths need denoising or a sequential matcher before they are trustworthy `p`s. The estimate then moves off the midpoint toward whichever mode `p` favors as `p` leaves ½. A particle filter kept from collapsing to one mode carries that `p` and emits the posterior mean for free — which is also why a blanket jump-guard backfires here, since suppressing the big jump deletes the second mode.
- **Spend a hold-out budget before a tuning budget.** Validate on a split shaped like the real test, and keep the harsh leave-whole-field-out as a pessimistic guardrail rather than the target. If you can construct a within-field hold-out, run it too as a second, less-pessimistic reading — but neither split is confirmed to match the hidden test geometry, so treat them as bracketing it. Run a seed sweep before trusting a small delta; an afternoon of hold-out budget beats a week of tuning.
- **Drop the confirmed dead end.** Cross-well surface reconstruction across fields (IDW / plane fit) fails badly, 13–129&nbsp;ft. Within-well typewell matching transfers; the cross-field offset does not. And the look-ahead is legal (Tiago thread), so whole-trace matching is allowed — use it.

<p style="background-color:#eef6ff;padding:16px;color:#111;font-size:15px;border:2px solid #9ec5ff;border-radius:6px">
<strong>If you keep one thing:</strong> the floor is not ~10. The board heads (~5.6) and my own smooth oracle (~3.0) both sit well below it, so the recoverable part runs much deeper than my selector reached. Spend your effort on median-well GR alignment and dip smoothing, and on the bimodal tail estimate the mode weight and predict the posterior mean rather than committing to one mode.
</p>

<a id="9"></a>
## 9. Fork-and-go harness: read your own ceiling and test your own features

Three self-contained helpers. Fork the notebook, drop in your own per-well predictions, and read your own numbers instead of taking mine.

`oracle_ceiling(wells)` takes your per-well `(true, s)` and returns the constant / line / smooth ceiling for *your* wells — your version of the §2 ladder.

`tail_concentration(wells)` fits the smooth oracle per well, computes each well's SSE, and returns the worst-10% share of the total — your version of the §5 concentration. Your number will differ from the figure in §5; that one is my train sample, this reads your tail.

`wall_test(D, feats)` is a field-*extrapolation* stress test by design, plus a leak check. Read it two ways. If a feature looks predictive in-sample and dies under `wall_test`, that is the check working — you were leaking. If a feature survives a within-field split but dies leave-field-out, that is field extrapolation, harder than what the competition scores, so don't read it as a verdict on the floor. Pair it with a within-field or random-row split so you see both the harsh number and the realistic one.

In [14]:
# ---- reusable harness: drop your own per-well (true, pred) in and read the ceiling / tail / wall ----
def oracle_ceiling(wells):
    # wells: list of dicts with 'true' (eval truth) and 's' (normalized MD 0..1). Returns the ceiling ladder.
    E = {k: [] for k in ["constant", "line", "smooth"]}
    for w in wells:
        t, s = np.asarray(w["true"], float), np.asarray(w["s"], float)
        E["constant"].append(t - t.mean())
        E["line"].append(t - np.polyval(np.polyfit(s, t, 1), s))
        E["smooth"].append(t - robfit(s, t, 6))
    return {k: rmse(np.concatenate(v)) for k, v in E.items()}

def tail_concentration(wells, frac=0.10):
    # worst-frac share of total smooth-oracle SSE — your version of the §5 tail.
    # your share will differ from the §5 figure (that's my train sample); this reads YOUR tail.
    sse = np.array([float(np.sum((np.asarray(w["true"], float) - robfit(np.asarray(w["s"], float), np.asarray(w["true"], float), 6))**2)) for w in wells])
    o = np.argsort(sse)[::-1]; k = max(1, int(len(sse)*frac))
    return round(float(sse[o[:k]].sum()/sse.sum()), 3)

def wall_test(D, feats, target="slope", n_fields=17, seed=42):
    # The HARD field-extrapolation test + a leak check, NOT a verdict on the task floor.
    # D: per-well dataframe with x,y + features + target. Field-grouped OOF R^2 (+ shuffle control).
    # Pair with a within-field or random-row split to get the realistic (less pessimistic) number.
    XY = D[["x","y"]].values; XYs = (XY-XY.mean(0))/(XY.std(0)+1e-9)
    fld = KMeans(n_fields, n_init=10, random_state=seed).fit(XYs).labels_
    y = D[target].values; rg = np.random.RandomState(seed); ys = rg.permutation(y)
    oof = np.zeros(len(D)); oos = np.zeros(len(D))
    for k in range(n_fields):
        tr, te = fld!=k, fld==k
        if te.sum()==0 or tr.sum()<15: oof[te]=oos[te]=y[tr].mean() if tr.sum() else y.mean(); continue
        f = lambda yy: HGB(max_iter=250,max_depth=3,learning_rate=.05,l2_regularization=2.,random_state=seed).fit(D.loc[tr,feats],yy[tr]).predict(D.loc[te,feats])
        oof[te], oos[te] = f(y), f(ys)
    r2 = lambda a,b: 1-np.sum((a-b)**2)/np.sum((a-a.mean())**2)
    return dict(field_oof_r2=round(r2(y,oof),3), shuffle_r2=round(r2(y,oos),3))

def posterior_mean(p, pred_lo, pred_hi):
    # squared-loss-optimal estimate on a two-mode well (pilkwang): p*lo + (1-p)*hi.
    # p = P(low mode) read from the two §4 shift-scan minima via a two-position softmax;
    # midpoint is the p=0.5 special case. apply only where a second mode is actually present.
    p = np.asarray(p, float)
    return p*np.asarray(pred_lo, float) + (1.0-p)*np.asarray(pred_hi, float)

# design notes: the oracle ladder is train-only / pooled row-RMSE; the reference lines are the REAL
# public LB (15.883 / ~7.2 / ~5.6); the field-OOF is a negative result on cheap features under a hard
# leave-field-out split; numbers shift a few hundredths across seeds/subsamples; no submission here.
print("ceiling:", {k: round(v,2) for k,v in oracle_ceiling(W).items()})
print("tail   :", tail_concentration(W), "(worst-10% share of SSE)")
print("wall   :", wall_test(D, feats))

<a id="10"></a>
## 10. Design choices and limitations

- **The oracle ladder is on train wells** (it needs the truth) and is *pooled* row-RMSE on a 250-well sample. The leaderboard reference lines are the real, honestly-scored public LB (carry-last 15.883, forks ~7.2, heads ~5.6). Bars and dashed lines are separate categories on a shared scale; the qualitative ordering carries the argument. The agreement between my ~16 flat oracle and the 15.883 baseline is a scale sanity check, not a validation — different well sets, same procedure.
- **The leave-field-out result is a negative result for the cheap features under a field-grouped split.** A fundamentally different signal could exist; the evidence says the cheap ones do not transfer across fields. It answers the harder cross-field question, not within-field learnability.
- **The bimodal-tail numbers are part measured, part borrowed.** The Milankovitch mechanism is souldrive's. The SSE concentration (worst-10% share) and the likelihood degeneracy are my own train-side measurements on the smooth-oracle residuals; the printed numbers are sample-specific and will shift on other data. The midpoint-versus-mode result is decision theory on an idealized 50/50 well. None of it is measured on the hidden test.
- **The public-LB fraction is approximate.** Roughly 26% of the ~200 hidden wells, on the order of 50; the private set decides medals; the host excluded one outlier private well.
- **On the retracted leak.** The visible `data/test/` example wells are byte-identical to the first three train wells, and an ANCC-cheat scored ~0.007 against that example set. That is what copies would do, and it is what I misread as the public LB being scored on copies. The real public LB is the hidden wells.
- **Subsample and seed.** Numbers shift a few hundredths across seeds and sample sizes; the ordering and signs are stable. Re-run with `N_WELLS = 773` to confirm.
- **No submission here.** This notebook scores nothing and is not meant to.

<a id="11"></a>
## 11. References and a question for you

The corrected thesis leans on more of the community than the first version did. Each of these is worth an upvote.

**The structure of the task**
- [franticXu — formation columns are derived from the typewell, not independent 3D surfaces](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction/discussion/708167): the one-degree-of-freedom result with the thickness evidence, and the point that `TVT` is cumulative vertical distance, not a single-layer thickness.
- [Tabish — Problem Breakdown](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction/discussion/708367): the typewell as a GR-vs-TVT lookup, shared subsequences of a master sequence, the piecewise-linear dip ("parallel jagged lines"), and the GR-rotation FFT denoising lever.
- [pilkwang — EDA: target-free alignment for TVT](https://www.kaggle.com/code/pilkwang/rogii-eda-target-free-alignment-for-tvt): the `TVT = −Z + S(X,Y) + b` framing and the GR-residual scale, with a careful look at leakage and overlap. In the comments here he also turned the §5 hedge into a calibrated posterior mean and supplied the seed-variance numbers cited in §7.

**The irreducible tail**
- [souldrive — the ±15 ft datum](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction/discussion/711878): the Milankovitch / bimodal-datum geology that §5 is built on. souldrive also publicly endorsed the first version of this notebook; the §5 synthesis is partly his.

**Starters and where a real pipeline's error lives**
- [Chris Deotte — EDA Starter](https://www.kaggle.com/code/cdeotte/eda-starter) and [XGB Starter](https://www.kaggle.com/code/cdeotte/xgb-starter-cv-15): the canonical orientation and a clean grouped-CV baseline.
- [mitchgansemer — GR features & outlier detection](https://www.kaggle.com/code/mitchgansemer/gr-features-outlier-detection-rogii-wellbore) and [drift-targeting](https://www.kaggle.com/code/mitchgansemer/drift-targeting-ncc-tree-based-rogii-wellbore): where the error concentrates on the long-tail wells, a useful counterpoint to §5 and §7.

**Rules and facts that settled things**
- The Tiago thread, where within-well look-ahead (using the full per-well GR trace) was confirmed legal, so whole-trace typewell matching is fair.
- Tucker (rank 2) described reaching about 5 pooled per-row RMSE using per-well data only. The LB heads sit ~5.6 (ranks 1–2); Tucker's ~5 is his own stated self-report, not a shared notebook, and it is the corroboration that the recoverable part runs below ~10.
- Ioannis M (rank 28) and the host posts that settled the test-set question: the visible `data/test/` is example data, replaced by the real hidden test.

If you want the floor and the field-extrapolation numbers on your own predictions, the helpers in §9 are self-contained — fork and point them at your OOF.

---
### Credits and a question for you

Thanks to the organizers and to franticXu, Tabish, pilkwang, souldrive, Chris Deotte and mitchgansemer, whose public work this builds on. License: Apache 2.0.

I got two things wrong in the first version, the leak and the wall, and I have tried to map the task more honestly here: a recoverable part (per-well offset plus piecewise dip, read off the typewell, where the median wells live) and an irreducible part (the bimodal datum on the tail, where the posterior mean beats any single mode). The floor is not ~10; the board heads at ~5.6 and the smooth oracle at ~3.0 both sit well below it. What I would like to see, and will run the harness on:

- a leak-free hold-out shaped like the real test, that I can validate against;
- a better GR alignment that pushes the median wells toward ~5;
- a reliable way to read the mode weight `p`, so the posterior mean lands only where a second mode is real.

Drop it in the comments. If you disagree, bring a hold-out and I'll run it.